In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:


#Import the required Libraries
import pandas as pd
import os


#Read the dataset using

#join path with file name
csv_path = os.path.join(path, "Q1_data.csv")
#convert the csv to a DataFrame
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
#Inspecting data
df.head()

In [ ]:
# Task 3: Write your code here:

#Display dataset information
df.info()

In [ ]:
# Task 4: Write your code here:

#Show statistical description
df.describe()

In [ ]:
# Task 5: Write your code here:

#Import the required Lib
import matplotlib.pyplot as plt
%matplotlib inline


#Plot the target distribution (delivery_time)
df['Delivery_Time'].hist(bins=30, edgecolor='black')

#Set the charts title
plt.title(f"Target Distribution ({'Delivery_Time'})")

#horizental axis label
plt.xlabel('Delivery Time')

#vertical axis label
plt.ylabel("Frequency")

plt.show()

In [ ]:
# Task 1: Write your code here:

#Drop the 'Order_ID' column from the data
df = df.drop(['Order_ID'], axis=1)

In [ ]:
# Task 2: Write your code here:

# Handle missing values appropriately

# check for missing values
missing_values = df.isnull().sum()

#print out col with missing values
print("Missing Values per Column:")
print(missing_values[missing_values > 0])


#Fill with mean (numerical)
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())

#Fill with mode (categorical)
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])


# check for missing values
df.info()

In [ ]:
# Task 3: Write your code here:

#Check and remove duplicates if any exist
duplicates = df.duplicated().sum()

#Remove duplicates if found
if duplicates > 0:
        print("Dropping Duplicates...")

        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
else:
        print("No Duplicate Samples Found.")



In [ ]:
# Task 4: Write your code here:
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Encode categorical variables
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

label_encoders = {}

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    # Apply fit_transform to encode the column
    df[col] = le.fit_transform(df[col])

df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

#Apply feature scaling for all features (standard)
print('data before scaling:\n', df) #show before scaling

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time") #scale numeric data (features only)

standard_scaler = StandardScaler() # Instantiate
df[numerical_cols]  = standard_scaler.fit_transform(df[numerical_cols] ) # fit/transform


print('data before scaling:\n', df) #show after scaling
df.head()

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not
Delivery_Time_counts = df['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(Delivery_Time_counts.index, Delivery_Time_counts.values, color='coral')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()



In [ ]:
# Task 1: Write your code here:
#Split the dataset into features (X) and target (y)

X, Y = df.drop('Delivery_Time',axis=1), df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

full_ratio = (Y.value_counts(normalize=True) * 100).sort_index()

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

#final average of mae
total_mae = 0

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]

    # print shapes
    print(f"Fold {fold}")

    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    print("-" * 30)

    #Train a RandomForest model
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    #Evaluate using MAE (Mean Absolute Error) ONLY

    # Predict and evaluate
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    total_mae += mae
    print(f"MAE:  ${mae:,.2f}")


    print("---Model trained!---\n")


#Print the averaged score across all folds
print('Averaged score across all folds =', f'${total_mae/5:,.2f}')

X_train.head()

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = ['Distance_km','Weather','Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)


plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])


plt.xlabel('Importance')
plt.title('Feature Importance')

plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
#Plot predicted delivery time histogram
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=30, edgecolor='black', color='blue')
plt.title('Predicted Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor


models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
    "CatBoost": CatBoostRegressor(verbose=0)
    }

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': []}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  for model_name, model in models.items():

        print(f"Training {model_name}...")

        # Train
        # fit
        model.fit(X_train, y_train)


        # Predict
        y_pred = model.predict(X_test)

        # Calculate metrics
        mse = sklearn_mae(y_test, y_pred)

        # Store results
        all_results[model_name]["mae"].append(mae)